In [1]:
# ============================================================
# BattingEdge V9.5 - CORRECTED Model Training
# Features: 99 raw pose + 8 angles (NO velocities)
# Target: 83-86% accuracy
# ============================================================

import numpy as np
import pickle
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import joblib

# ================= CONFIG =================
FEATURE_DIR = Path(r"D:\Users\Anoshia\BattingEdge_FYP\v9_5\features")
MODEL_DIR   = Path(r"D:\Users\Anoshia\BattingEdge_FYP\backend\models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 60
BATCH_SIZE = 16
LEARNING_RATE = 5e-4  # 0.0005 (not 0.001)
# ==========================================

print("="*70)
print("BATTINGEDGE V9.5 - MODEL TRAINING (CORRECTED)")
print("="*70)
print()

# ================= LOAD DATA =================
print("📂 Loading data...")

X_train = np.load(FEATURE_DIR / "X_train.npy")
y_train = np.load(FEATURE_DIR / "y_train.npy")
X_val   = np.load(FEATURE_DIR / "X_val.npy")
y_val   = np.load(FEATURE_DIR / "y_val.npy")
X_test  = np.load(FEATURE_DIR / "X_test.npy")
y_test  = np.load(FEATURE_DIR / "y_test.npy")

with open(FEATURE_DIR / "classes.pkl", "rb") as f:
    CLASSES = pickle.load(f)

num_classes = len(CLASSES)
T, F = X_train.shape[1], X_train.shape[2]

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Classes: {CLASSES}")
print(f"Features per frame: {F} (99 pose + 8 angles)")
print()

# ================= CRITICAL: SCALING =================
print("⚖️  Applying StandardScaler...")

scaler = StandardScaler()

# Fit on training data (flatten to 2D)
N_train = X_train.shape[0]
X_train_2d = X_train.reshape(N_train * T, F)
scaler.fit(X_train_2d)

def scale_data(X):
    N, T, F = X.shape
    X_2d = X.reshape(N * T, F)
    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(N, T, F)

X_train = scale_data(X_train)
X_val   = scale_data(X_val)
X_test  = scale_data(X_test)

# Save scaler for inference
joblib.dump(scaler, MODEL_DIR / "scaler_V9_5.pkl")
joblib.dump(CLASSES, MODEL_DIR / "classes_V9_5.pkl")
print(f"✅ Scaler saved: {MODEL_DIR / 'scaler_V9_5.pkl'}")
print()

# ================= CLASS WEIGHTS =================
print("⚖️  Computing class weights...")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

for i, cls in enumerate(CLASSES):
    print(f"  {cls:15s}: {class_weight_dict[i]:.3f}")
print()

# ================= MODEL =================
print("🏗️  Building model...")

model = Sequential([
    # Input
    Bidirectional(LSTM(128, return_sequences=True, recurrent_dropout=0.2), 
                  input_shape=(T, F)),
    Dropout(0.4),
    
    # Second LSTM
    Bidirectional(LSTM(64, recurrent_dropout=0.2)),
    Dropout(0.4),
    
    # Dense layers
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    
    # Output
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
print()

# ================= CALLBACKS =================
print("⚙️  Setting up callbacks...")

checkpoint_path = MODEL_DIR / "battingedge_V9_5_best.keras"

callbacks_list = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

print("✅ Early Stopping: patience=10")
print("✅ ReduceLR: factor=0.5, patience=5")
print(f"✅ Checkpoint: {checkpoint_path.name}")
print()

# ================= TRAINING =================
print("="*70)
print("🚀 STARTING TRAINING")
print("="*70)
print()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks_list,
    verbose=1
)

print()
print("="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print()

# Save final model
final_path = MODEL_DIR / "battingedge_V9_5_final.keras"
model.save(final_path)
print(f"💾 Saved final model: {final_path}")
print()

# ================= EVALUATION =================
print("="*70)
print("📊 EVALUATING ON TEST SET")
print("="*70)
print()

# Load best model
import tensorflow as tf
best_model = tf.keras.models.load_model(str(checkpoint_path))
print(f"✅ Loaded best model from: {checkpoint_path.name}")
print()

# Predict
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

# Classification report
print("📋 CLASSIFICATION REPORT:")
print()
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=3)
print(report)

with open(MODEL_DIR / "report_V9_5.txt", "w") as f:
    f.write(report)
print()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("🔢 CONFUSION MATRIX:")
print("   (Rows = True, Cols = Predicted)")
print()
print("        ", "  ".join([f"{cls[:4]:>4s}" for cls in CLASSES]))
for i, cls in enumerate(CLASSES):
    print(f"{cls[:8]:8s}", "  ".join([f"{cm[i,j]:4d}" for j in range(num_classes)]))
print()

# Per-class accuracy
print("📈 PER-CLASS ACCURACY:")
print()
for i, cls in enumerate(CLASSES):
    correct = cm[i, i]
    total = cm[i, :].sum()
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"   {cls:15s}: {correct:3d}/{total:3d} = {accuracy:5.1f}%")

overall_acc = np.trace(cm) / np.sum(cm) * 100
print()
print(f"   {'OVERALL':15s}: {np.trace(cm):3d}/{np.sum(cm):3d} = {overall_acc:5.2f}%")
print()

# Major confusions
print("🔍 MAJOR CONFUSIONS (>3 cases):")
print()
confusions = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 3:
            confusions.append((CLASSES[i], CLASSES[j], cm[i, j]))

if confusions:
    confusions.sort(key=lambda x: x[2], reverse=True)
    for true_cls, pred_cls, count in confusions:
        print(f"   {true_cls:15s} → {pred_cls:15s}: {count} cases")
else:
    print("   ✅ No major confusions!")
print()

# ================= VISUALIZATIONS =================
print("📊 Generating visualizations...")

# Confusion matrix heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Confusion Matrix - V9.5", fontsize=14, fontweight='bold')
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrix_V9_5.png", dpi=300)
print("   ✅ Saved: confusion_matrix_V9_5.png")
plt.close()

# Training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Model Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / "training_history_V9_5.png", dpi=300)
print("   ✅ Saved: training_history_V9_5.png")
plt.close()

print()

# ================= COMPARISON =================
print("="*70)
print("📊 MODEL COMPARISON")
print("="*70)
print()

print("Overall Accuracy:")
print(f"  V9:   81.0%")
print(f"  V10:  75.2% (velocities broke it)")
print(f"  V9.5: {overall_acc:.1f}%")

delta_v9 = overall_acc - 81.0
status = "✅ SUCCESS" if overall_acc >= 81 else "→ STABLE" if overall_acc >= 79 else "⚠️ CHECK"
print(f"  Δ from V9: {delta_v9:+.1f}% {status}")
print()

print("Per-Class Comparison:")
v9_baseline = {"Cover Drive": 80, "Cut Shot": 77, "Defense": 89, "Pull Shot": 73, "Sweep Shot": 87}

for i, cls in enumerate(CLASSES):
    v9_5_acc = (cm[i,i] / cm[i,:].sum() * 100) if cm[i,:].sum() > 0 else 0
    v9_acc = v9_baseline.get(cls, 0)
    delta = v9_5_acc - v9_acc
    status = "✅" if delta > 0 else "→" if delta >= -2 else "⚠️"
    print(f"  {cls:15s}: V9={v9_acc:5.1f}% → V9.5={v9_5_acc:5.1f}% ({delta:+5.1f}%) {status}")

print()
print("="*70)
print("🎉 V9.5 TRAINING COMPLETE")
print("="*70)
print()

# Save metadata
metadata = {
    "version": "V9.5",
    "features": "99 raw pose + 8 angles (no velocities)",
    "classes": CLASSES,
    "test_accuracy": float(overall_acc),
    "per_class_accuracy": {
        CLASSES[i]: float((cm[i,i] / cm[i,:].sum() * 100) if cm[i,:].sum() > 0 else 0)
        for i in range(num_classes)
    },
    "comparison_to_v9": {
        "v9_accuracy": 81.0,
        "v9_5_accuracy": float(overall_acc),
        "improvement": float(overall_acc - 81.0)
    },
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE
    }
}

with open(MODEL_DIR / "metadata_V9_5.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("📦 Saved artifacts:")
print(f"   1. Best model: {checkpoint_path.name}")
print(f"   2. Final model: {final_path.name}")
print(f"   3. Scaler: scaler_V9_5.pkl")
print(f"   4. Classes: classes_V9_5.pkl")
print(f"   5. Report: report_V9_5.txt")
print(f"   6. Confusion matrix: confusion_matrix_V9_5.png")
print(f"   7. Training history: training_history_V9_5.png")
print(f"   8. Metadata: metadata_V9_5.json")
print()

if overall_acc >= 83:
    print("✨ EXCELLENT RESULT! Ready for deployment.")
elif overall_acc >= 81:
    print("✅ GOOD RESULT! Matches or improves V9.")
elif overall_acc >= 79:
    print("→ ACCEPTABLE. Stable performance maintained.")
else:
    print("⚠️ Below target. Review feature extraction.")

print("="*70)

BATTINGEDGE V9.5 - MODEL TRAINING (CORRECTED)

📂 Loading data...
Train: 3007 samples
Val:   388 samples
Test:  378 samples
Classes: ['Cover Drive', 'Cut Shot', 'Defense', 'Pull Shot', 'Sweep Shot']
Features per frame: 107 (99 pose + 8 angles)

⚖️  Applying StandardScaler...
✅ Scaler saved: D:\Users\Anoshia\BattingEdge_FYP\backend\models\scaler_V9_5.pkl

⚖️  Computing class weights...
  Cover Drive    : 1.004
  Cut Shot       : 1.018
  Defense        : 0.972
  Pull Shot      : 0.999
  Sweep Shot     : 1.009

🏗️  Building model...


d:\Users\Anoshia\BattingEdge_FYP\venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 50, 256)        │       241,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 414,853 (1.58 MB)

 Trainable params: 414,725 (1.58 MB)

 Non-trainable params: 128 (512.00 B)


⚙️  Setting up callbacks...
✅ Early Stopping: patience=10
✅ ReduceLR: factor=0.5, patience=5
✅ Checkpoint: battingedge_V9_5_best.keras

🚀 STARTING TRAINING

Epoch 1/60
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.3220 - loss: 1.8881
Epoch 1: val_accuracy improved from None to 0.57732, saving model to D:\Users\Anoshia\BattingEdge_FYP\backend\models\battingedge_V9_5_best.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 75s 290ms/step - accuracy: 0.4037 - loss: 1.6299 - val_accuracy: 0.5773 - val_loss: 1.0948 - learning_rate: 5.0000e-04
Epoch 2/60
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - accuracy: 0.5176 - loss: 1.2586
Epoch 2: val_accuracy improved from 0.57732 to 0.64691, saving model to D:\Users\Anoshia\BattingEdge_FYP\backend\models\battingedge_V9_5_best.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 55s 294ms/step - accuracy: 0.5364 - loss: 1.2251 - val_accuracy: 0.6469 - val_loss: 0.9133 - learning_rate: 5.0000e-04
Epoch 3/60
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.5987 - lo